<a href="https://colab.research.google.com/github/narame7/UOS-FootballDataAnalytics-Tutorial/blob/main/Week%202/1-load-statistic-data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 통계 데이터 크롤링

In [ ]:
!pip install soccerdata

In [ ]:
import io
import re

import pandas as pd
import requests
import soccerdata as sd

## 필요한 라이브러리 import

## [ClubElo](http://clubelo.com/)
- 전 세계 클럽의 Elo 점수(팀 전력 지수)를 계산해 순위를 매기는 사이트.
- 원래는 `api.clubelo.com` API가 있었지만 2026년 9월 현재 서버가 응답하지 않아(502), 웹페이지의 HTML 표를 `pandas.read_html`로 직접 읽어 옵니다.
- `read_html`은 페이지 안의 모든 `<table>`을 DataFrame 목록으로 돌려주므로, 원하는 표를 골라내는 과정이 필요합니다.

In [ ]:
import time

# 많은 사이트가 브라우저가 아닌 요청(파이썬 기본 User-Agent)을 차단하므로 브라우저처럼 보이는 헤더를 붙입니다.
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0 Safari/537.36'}

def get_html(url, retries=3):
    """웹페이지 HTML을 받아옵니다. 서버가 연결을 끊거나 오류를 내면 잠시 쉬었다가 다시 시도합니다."""
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()  # 200이 아니면 오류
            if not response.text.strip():  # 봇 차단 페이지는 200이면서 본문이 비어 있는 경우가 있음
                raise requests.RequestException('빈 응답')
            return response.text
        except requests.RequestException as e:
            if attempt == retries - 1:
                raise
            print(f'요청 실패({e.__class__.__name__}), 5초 후 재시도합니다...')
            time.sleep(5)

def read_tables(url):
    """웹페이지 안의 모든 표를 DataFrame 목록으로 돌려줍니다."""
    return pd.read_html(io.StringIO(get_html(url)))

### 리그별 현재 Elo 순위
- 순위 페이지에는 리그마다 `Club`, `Elo` 두 열짜리 표가 하나씩 있습니다. `Club` 열은 `'2 ARSArsenal'`처럼 세계 순위 + 팀 코드 + 팀 이름이 붙어 있어 정규식으로 나눕니다.

In [ ]:
elo_pages = read_tables('http://clubelo.com/Ranking')
elo_tables = [t for t in elo_pages if list(t.columns) == ['Club', 'Elo']]
print(len(elo_pages), '개 표 중 리그 순위표', len(elo_tables), '개')

def clean_elo_table(table):
    parts = table['Club'].str.extract(r'^(\d+)\s+([A-Z]{3})(.+)$')  # 세계 순위, 팀 코드, 팀 이름
    out = pd.DataFrame({'world_rank': parts[0], 'code': parts[1], 'club': parts[2], 'elo': table['Elo']})
    out = out.dropna(subset=['world_rank'])  # 'Level 1 (20 teams)' 같은 구분 행 제거
    return out.astype({'world_rank': int, 'elo': int}).reset_index(drop=True)

england_elo = next(clean_elo_table(t) for t in elo_tables if t['Club'].str.contains('Arsenal').any())
england_elo.head(20)  # 잉글랜드 1~4부 팀들의 Elo

In [ ]:
world_elo = pd.concat([clean_elo_table(t) for t in elo_tables]).sort_values('world_rank').reset_index(drop=True)
world_elo.head(20)  # 세계 Elo 상위 20팀

### 특정 팀의 최근 경기별 Elo 변화
- 팀 페이지(`clubelo.com/팀이름`)에는 최근 경기마다 상대, 결과, Elo 증감, 새 순위가 표로 나옵니다.

In [ ]:
liverpool_pages = read_tables('http://clubelo.com/Liverpool')
liverpool_recent = next(t for t in liverpool_pages if 'New Elo' in t.columns)
liverpool_recent = liverpool_recent.rename(columns={liverpool_recent.columns[0]: 'Date'})
liverpool_recent[['Date', 'H/A', 'Opponent', 'FT', 'Elo +/-', 'New Elo', 'New Rank']]

## [Fantasy Premier League API](https://fantasy.premierleague.com/)
- 프리미어리그 공식 판타지 게임의 API. 로그인이나 키 없이 JSON으로 선수·팀·경기 데이터를 줍니다. 축구 데이터 입문에서 가장 많이 쓰는 공개 API 중 하나입니다.
- 게임용 지표(판타지 점수, 가격, 선택 비율)와 실제 경기 기록(득점, 도움, 출전 시간, xG, xA 등)이 함께 들어 있습니다.
- 주요 주소
  - `bootstrap-static/` : 팀, 포지션, 선수 시즌 누적 기록, 게임위크 정보 (한 번에 다 옴)
  - `fixtures/` : 시즌 전체 경기 일정과 결과
  - `element-summary/{선수 id}/` : 특정 선수의 경기별 기록

In [ ]:
FPL = 'https://fantasy.premierleague.com/api'

def get_json(url, retries=3):
    """JSON API를 호출해 파이썬 dict/list로 돌려줍니다. 연결이 끊기면 잠시 쉬었다가 다시 시도합니다."""
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            if attempt == retries - 1:
                raise
            print(f'요청 실패({e.__class__.__name__}), 5초 후 재시도합니다...')
            time.sleep(5)

fpl_data = get_json(f'{FPL}/bootstrap-static/')
print('최상위 키:', list(fpl_data.keys()))
print('선수 수:', len(fpl_data['elements']), '| 팀 수:', len(fpl_data['teams']), '| 게임위크 수:', len(fpl_data['events']))

### 팀과 포지션 정보
- 선수 데이터에는 팀과 포지션이 숫자 id로만 들어 있어서, 이름표를 붙이려면 `teams`와 `element_types`를 따로 꺼내 두어야 합니다.

In [ ]:
fpl_teams = pd.DataFrame(fpl_data['teams'])[['id', 'name', 'short_name', 'strength', 'strength_attack_home', 'strength_attack_away', 'strength_defence_home', 'strength_defence_away']]
fpl_positions = pd.DataFrame(fpl_data['element_types'])[['id', 'singular_name_short', 'singular_name']]
print(fpl_positions.to_string(index=False))
fpl_teams

### 선수 시즌 누적 기록
- 열이 100개가 넘으므로 자주 쓰는 것만 골라 이름을 붙입니다. 숫자가 문자열로 들어 있는 열(`expected_goals` 등)은 `pd.to_numeric`으로 바꿔야 정렬과 계산이 됩니다. 가격(`now_cost`)은 0.1£m 단위라 10으로 나눕니다.

In [ ]:
players_raw = pd.DataFrame(fpl_data['elements'])
print('전체 열 수:', players_raw.shape[1])

players = players_raw[['id', 'web_name', 'team', 'element_type', 'now_cost', 'minutes', 'goals_scored', 'assists', 'clean_sheets',
                       'expected_goals', 'expected_assists', 'total_points', 'ict_index', 'selected_by_percent', 'status']].copy()
players = players.merge(fpl_teams[['id', 'name']].rename(columns={'id': 'team', 'name': 'team_name'}), on='team')
players = players.merge(fpl_positions[['id', 'singular_name_short']].rename(columns={'id': 'element_type', 'singular_name_short': 'position'}), on='element_type')
for col in ['expected_goals', 'expected_assists', 'ict_index', 'selected_by_percent']:
    players[col] = pd.to_numeric(players[col])
players['price'] = players['now_cost'] / 10  # £m 단위
players = players.drop(columns=['team', 'element_type', 'now_cost'])
players.head()

### 득점·xG 순위
- 실제 득점과 기대 득점(xG)을 나란히 놓으면 "운이 좋았던" 선수와 "기회는 많이 만들었지만 못 넣은" 선수가 보입니다.

In [ ]:
top_scorers = players.sort_values(['goals_scored', 'expected_goals'], ascending=False).head(15)
top_scorers[['web_name', 'team_name', 'position', 'minutes', 'goals_scored', 'expected_goals', 'assists', 'expected_assists']]

In [ ]:
# 출전 시간이 어느 정도 되는 선수 중에서 xG 대비 득점 차이가 큰 선수 (양수: 기대보다 많이 넣음, 음수: 기대보다 못 넣음)
regulars = players[players['minutes'] >= 180].copy()
regulars['goals_minus_xg'] = regulars['goals_scored'] - regulars['expected_goals']
regulars.sort_values('goals_minus_xg', ascending=False)[['web_name', 'team_name', 'position', 'minutes', 'goals_scored', 'expected_goals', 'goals_minus_xg']].head(10)

### 팀별 합계와 포지션별 비교

In [ ]:
team_totals = players.groupby('team_name')[['goals_scored', 'expected_goals', 'assists', 'expected_assists', 'total_points']].sum().round(2)
team_totals.sort_values('goals_scored', ascending=False)

In [ ]:
# 포지션별로 90분당 판타지 점수와 가격 대비 점수를 비교
regulars['points_per_90'] = regulars['total_points'] / regulars['minutes'] * 90
regulars['points_per_price'] = regulars['total_points'] / regulars['price']
regulars.groupby('position')[['price', 'total_points', 'points_per_90', 'points_per_price']].mean().round(2)

In [ ]:
# 가장 많이 선택된 선수 15명: 판타지 유저들의 '인기 순위'
players.sort_values('selected_by_percent', ascending=False)[['web_name', 'team_name', 'position', 'price', 'total_points', 'selected_by_percent']].head(15)

### 경기 일정과 결과
- `fixtures/`는 380경기 전체를 줍니다. 팀 id를 이름으로 바꾸고, 끝난 경기만 골라 봅니다. `team_h_difficulty`, `team_a_difficulty`는 FPL이 매긴 상대 난이도(1~5)입니다.

In [ ]:
fixtures = pd.DataFrame(get_json(f'{FPL}/fixtures/'))
team_names = fpl_teams.set_index('id')['name']
fixtures['home'] = fixtures['team_h'].map(team_names)
fixtures['away'] = fixtures['team_a'].map(team_names)
fixtures['kickoff_time'] = pd.to_datetime(fixtures['kickoff_time'])

finished = fixtures[fixtures['finished']]
print('끝난 경기:', len(finished), '/', len(fixtures))
finished[['event', 'kickoff_time', 'home', 'team_h_score', 'team_a_score', 'away', 'team_h_difficulty', 'team_a_difficulty']].tail(10)

### 특정 선수의 경기별 기록
- `element-summary/{id}/`의 `history`에 게임위크마다 출전 시간, 득점, xG, 판타지 점수, 당시 가격이 들어 있습니다.

In [ ]:
player = players.sort_values('total_points', ascending=False).iloc[0]  # 판타지 점수 1위 선수
print(player['web_name'], '|', player['team_name'], '|', player['position'])

history = pd.DataFrame(get_json(f'{FPL}/element-summary/{int(player["id"])}/')['history'])
history['opponent'] = history['opponent_team'].map(team_names)
history[['round', 'opponent', 'was_home', 'minutes', 'goals_scored', 'assists', 'expected_goals', 'expected_assists', 'total_points', 'value']]

## [프리미어리그 공식 사이트 API](https://www.premierleague.com/)
- premierleague.com 페이지가 내부적으로 호출하는 API(`footballapi.pulselive.com`)입니다. 키 없이 JSON을 주지만 **공식 문서가 없는 비공식 API**라서 주소나 구조가 예고 없이 바뀔 수 있습니다.
- 브라우저 개발자 도구의 네트워크 탭에서 사이트가 어떤 요청을 보내는지 보고 찾아낸 주소들입니다. 많은 사이트의 "숨은 API"는 이런 식으로 찾습니다.
- 시즌은 `compSeasons` id로 지정합니다(2026/27 = 841). 아래 첫 셀에서 최근 시즌 id를 확인할 수 있습니다.

In [ ]:
PL = 'https://footballapi.pulselive.com/football'

seasons = get_json(f'{PL}/competitions/1/compseasons?pageSize=5')['content']
pd.DataFrame(seasons)[['id', 'label']].astype({'id': int})

### 순위표
- 순위표 JSON은 팀마다 `overall`, `home`, `away` 기록과 홈구장 정보가 중첩된 dict로 들어 있어서, 필요한 값만 꺼내 평평한 표로 만듭니다.

In [ ]:
SEASON_ID = 841  # 2026/27

standings_json = get_json(f'{PL}/standings?compSeasons={SEASON_ID}&altIds=true')
rows = []
for entry in standings_json['tables'][0]['entries']:
    overall, home, away = entry['overall'], entry['home'], entry['away']
    rows.append({
        'position': entry['position'],
        'team': entry['team']['name'],
        'played': overall['played'], 'won': overall['won'], 'drawn': overall['drawn'], 'lost': overall['lost'],
        'GF': overall['goalsFor'], 'GA': overall['goalsAgainst'], 'GD': overall['goalsDifference'], 'points': overall['points'],
        'home_points': home['points'], 'away_points': away['points'],
        'ground': entry['ground']['name'], 'capacity': entry['ground'].get('capacity'),  # 일부 구장은 수용 인원이 없음
    })
pl_standings = pd.DataFrame(rows)
pl_standings

### 선수 기록 순위
- `stats/ranked/players/{지표}` 주소로 지표별 순위를 받을 수 있습니다. 지표 이름 예: `goals`(득점), `goal_assist`(도움), `clean_sheet`(무실점), `total_scoring_att`(슈팅), `touches`(볼 터치).

In [ ]:
def pl_player_ranking(stat, n=10):
    """지표별 선수 순위를 DataFrame으로 돌려줍니다."""
    content = get_json(f'{PL}/stats/ranked/players/{stat}?compSeasons={SEASON_ID}&pageSize={n}&altIds=true')['stats']['content']
    return pd.DataFrame([{
        'rank': i + 1,
        'player': item['owner']['name']['display'],
        'team': item['owner'].get('currentTeam', {}).get('name'),
        'nationality': item['owner'].get('nationalTeam', {}).get('country'),
        'position': item['owner'].get('info', {}).get('positionInfo'),
        stat: item['value'],
    } for i, item in enumerate(content)])

pl_player_ranking('goals', 15)

In [ ]:
# 슈팅 상위 선수와 득점을 합쳐 슈팅당 득점(결정력) 비교
shots = pl_player_ranking('total_scoring_att', 30)
goals = pl_player_ranking('goals', 100)[['player', 'goals']]
conversion = shots.merge(goals, on='player', how='left').fillna({'goals': 0})
conversion['goals_per_shot'] = (conversion['goals'] / conversion['total_scoring_att']).round(3)
conversion[['player', 'team', 'position', 'total_scoring_att', 'goals', 'goals_per_shot']]

### 경기 결과와 관중 수
- `fixtures` 주소는 페이지 단위로 경기를 줍니다(`statuses=C`는 끝난 경기). 경기장과 관중 수가 함께 오므로 관중 순위도 만들 수 있습니다.

In [ ]:
fixtures_json = get_json(f'{PL}/fixtures?compSeasons={SEASON_ID}&pageSize=50&sort=desc&statuses=C&altIds=true')
print('끝난 경기:', fixtures_json['pageInfo']['numEntries'])

results = pd.DataFrame([{
    'gameweek': f['gameweek']['gameweek'],
    'kickoff': f['kickoff']['label'],
    'home': f['teams'][0]['team']['name'], 'home_score': f['teams'][0]['score'],
    'away_score': f['teams'][1]['score'], 'away': f['teams'][1]['team']['name'],
    'ground': f['ground']['name'], 'attendance': f.get('attendance'),
} for f in fixtures_json['content']])
results.sort_values('attendance', ascending=False).head(10)

## [Wikipedia](https://en.wikipedia.org/wiki/2026%E2%80%9327_Premier_League)
- 위키백과의 시즌 문서에는 순위표, 경기장, 감독·주장, 득점 순위 같은 표가 정리되어 있습니다. 봇 차단이 없어 어디서든 안정적으로 읽을 수 있습니다.
- 표 안에 `[a]`, `[16]` 같은 각주 표시가 섞여 있어서 정리하는 과정이 필요합니다. 실제 크롤링에서 늘 만나는 "지저분한 표 정리" 연습입니다.

In [ ]:
WIKI_URL = 'https://en.wikipedia.org/wiki/2026%E2%80%9327_Premier_League'  # '–'(en dash)는 URL에서 %E2%80%93
wiki_tables = read_tables(WIKI_URL)
print(len(wiki_tables), '개 표')

def strip_footnotes(df):
    """열 이름과 문자열 값에 붙은 각주 표시([a], [16] 등)를 지웁니다."""
    df = df.copy()
    df.columns = [re.sub(r'\[.*?\]', '', str(c)).strip() for c in df.columns]
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].str.replace(r'\[.*?\]', '', regex=True).str.strip()
    return df

def find_table(tables, required_cols):
    """필요한 열을 모두 가진 첫 번째 표를 찾아 각주를 정리해 돌려줍니다."""
    for t in tables:
        cols = [re.sub(r'\[.*?\]', '', str(c)).strip() for c in t.columns]
        if all(c in cols for c in required_cols):
            return strip_footnotes(t)
    raise ValueError(f'{required_cols} 열을 가진 표가 없습니다')

### 리그 순위표

In [ ]:
wiki_table = find_table(wiki_tables, ['Pos', 'Team', 'Pts'])
wiki_table['Team'] = wiki_table['Team'].str.replace(r'\s*\(.*\)$', '', regex=True)  # 'Arsenal (C)' 같은 표시 제거
wiki_table

### 경기장과 수용 인원

In [ ]:
stadiums = find_table(wiki_tables, ['Team', 'Stadium', 'Capacity'])
stadiums['Capacity'] = stadiums['Capacity'].astype(str).str.replace(',', '').astype(int)  # '36,887' -> 36887
stadiums.sort_values('Capacity', ascending=False)

### 감독과 주장

In [ ]:
staff = find_table(wiki_tables, ['Team', 'Manager', 'Captain'])
staff[['Team', 'Manager', 'Captain', 'Kit manufacturer']]

### 득점 순위

In [ ]:
scorers = find_table(wiki_tables, ['Rank', 'Player', 'Club', 'Goals'])
scorers

### 표 합치기
- 순위표와 경기장 표를 팀 이름으로 합쳐, 승점과 경기장 크기를 나란히 봅니다. 서로 다른 표의 팀 이름 표기가 같아야 합쳐지므로, 안 합쳐진 팀이 있는지 꼭 확인합니다.

In [ ]:
merged = pd.merge(wiki_table[['Pos', 'Team', 'Pld', 'GF', 'GA', 'Pts']], stadiums[['Team', 'Stadium', 'Capacity']], on='Team', how='left')
print('경기장 정보가 안 붙은 팀:', merged[merged['Capacity'].isna()]['Team'].tolist())
merged

## [Understat](https://understat.com/)

- 자체 xG 모델과 여러 기록이 있는 통계 사이트
- [soccerdata](https://soccerdata.readthedocs.io/en/latest/datasources/Understat.html) 라이브러리의 Understat 리더를 사용합니다. 사이트 구조가 바뀌면 라이브러리도 같이 업데이트해야 하므로, 오류가 나면 먼저 `pip install -U soccerdata`로 최신 버전인지 확인하세요.

In [ ]:
us = sd.Understat(leagues="ENG-Premier League", seasons="2026/2027")

### 경기 일정과 링크 불러오기

In [ ]:
schedule = us.read_schedule().reset_index()
us_match_links = schedule['url'].tolist()
schedule[['date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'url']].head()

### 팀 순위 보기

In [ ]:
team_match = us.read_team_match_stats().reset_index()

# 경기 단위 기록을 팀 단위(홈/원정)로 풀어서 순위표 만들기
cols = ['team', 'points', 'goals_for', 'goals_against', 'xg', 'xga']
home_rows = team_match[['home_team', 'home_points', 'home_goals', 'away_goals', 'home_xg', 'away_xg']].set_axis(cols, axis=1).assign(venue='Home')
away_rows = team_match[['away_team', 'away_points', 'away_goals', 'home_goals', 'away_xg', 'home_xg']].set_axis(cols, axis=1).assign(venue='Away')
team_rows = pd.concat([home_rows, away_rows], ignore_index=True)

def league_table(df):
    table = df.groupby('team').agg(
        M=('points', 'size'), Pts=('points', 'sum'),
        GF=('goals_for', 'sum'), GA=('goals_against', 'sum'),
        xG=('xg', 'sum'), xGA=('xga', 'sum'),
    )
    table['GD'] = table['GF'] - table['GA']
    return table.sort_values(['Pts', 'GD', 'GF'], ascending=False)

tables = [league_table(team_rows),
          league_table(team_rows[team_rows['venue'] == 'Home']),
          league_table(team_rows[team_rows['venue'] == 'Away'])]

In [ ]:
tables[0] # 전체 순위, 홈 경기 순위, 원정 경기 순위

### 팀 내 순위 보기

In [ ]:
player_stats = us.read_player_season_stats().reset_index()

In [ ]:
player_stats['team'].unique()

In [ ]:
everton = player_stats[player_stats['team'] == 'Everton']
everton[['player', 'position', 'matches', 'minutes', 'goals', 'xg', 'assists', 'xa', 'shots', 'key_passes']].sort_values('xg', ascending=False).head()

### 경기 정보 보기

In [ ]:
first_match = schedule.iloc[0]
us_match = us.read_player_match_stats(match_id=int(first_match['game_id'])).reset_index()

In [ ]:
us_match['team'].unique() # home away 구분

In [ ]:
us_match_df = us_match[us_match['team'] == first_match['home_team']] # 첫번째 경기의 홈 팀 정보

In [ ]:
us_match_df.head()

### 여러 경기의 슈팅 데이터 보기

In [ ]:
match_ids = schedule['game_id'].iloc[:5].astype(int).tolist() # 처음 5경기
us_shots = us.read_shot_events(match_id=match_ids)

In [ ]:
us_shots.reset_index()['game'].value_counts()

In [ ]:
us_shots.head()

### 시즌 누적 데이터 불러오기

In [ ]:
player_stats.sort_values('xg', ascending=False).head(10) # 시즌 xG 상위 10명

### 특정 팀의 경기 데이터 불러오기

In [ ]:
team_name = 'Aston Villa'
us_team_data = team_match[(team_match['home_team'] == team_name) | (team_match['away_team'] == team_name)]
us_team_data[['date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'home_ppda', 'away_ppda']].head() # 특정 팀(aston villa)의 경기 기록

### [SoccerData](https://soccerdata.readthedocs.io/en/latest/)

In [33]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_context("notebook")
sns.set_style("whitegrid")

#### Home team advantage in the Italian Serie A

In [ ]:
# We all know sports teams have an advantage when playing at home. Here’s a look at home team advantage for 5 years of the Serie A.
try:
    seriea_hist = sd.MatchHistory("ITA-Serie A", range(2018, 2023))
    games = seriea_hist.read_games()
except Exception as e:
    # football-data.co.uk 서버가 응답하지 않으면 아래 홈 어드밴티지 셀들은 건너뜁니다.
    games = None
    print('football-data.co.uk에서 데이터를 받지 못했습니다. 잠시 후 다시 시도해 보세요.', repr(e))
games.sample(5) if games is not None else None

In [37]:
def home_away_results(games: pd.DataFrame):
    """Returns aggregated home/away results per team"""
    res = pd.melt(
        games.reset_index(),
        id_vars=["date", "FTR"],
        value_name="team",
        var_name="is_home",
        value_vars=["home_team", "away_team"],
    )

    res.is_home = res.is_home.replace(["home_team", "away_team"], ["Home", "Away"])
    res["win"] = res["lose"] = res["draw"] = 0
    res.loc[(res["is_home"] == "Home") & (res["FTR"] == "H"), "win"] = 1
    res.loc[(res["is_home"] == "Away") & (res["FTR"] == "A"), "win"] = 1
    res.loc[(res["is_home"] == "Home") & (res["FTR"] == "A"), "lose"] = 1
    res.loc[(res["is_home"] == "Away") & (res["FTR"] == "H"), "lose"] = 1
    res.loc[res["FTR"] == "D", "draw"] = 1

    groups = res.groupby(["team", "is_home"])
    win = groups.win.agg(["sum", "mean"]).rename(columns={"sum": "n_win", "mean": "win_pct"})
    loss = groups.lose.agg(["sum", "mean"]).rename(columns={"sum": "n_lose", "mean": "lose_pct"})
    draw = groups.draw.agg(["sum", "mean"]).rename(columns={"sum": "n_draw", "mean": "draw_pct"})

    res = pd.concat([win, loss, draw], axis=1)
    return res

In [ ]:
results = home_away_results(games) if games is not None else None
results.head(6) if results is not None else None

In [ ]:
# The overall picture shows most teams have a clear advantage at home:
if results is not None:
    g = sns.FacetGrid(results.reset_index(), hue="team", palette="Set2", height=6, aspect=0.5)
    g.map(sns.pointplot, "is_home", "win_pct", order=["Away", "Home"])
    g.set_axis_labels("", "win %");

In [ ]:
# But there are a few exceptions
if results is not None:
    g = sns.FacetGrid(results.reset_index(), col="team", col_wrap=5)
    g.map(sns.pointplot, "is_home", "win_pct", order=["Away", "Home"])
    g.set_axis_labels("", "win %");